# API Gateway & Service Discovery

## 🧠 Mental Model: API Gateway

> **An API Gateway is the front door of a building. The outside world knows ONE address.
> Behind it, many different rooms (services) exist. The receptionist (gateway) decides
> which room to send each visitor to, checks IDs, logs arrivals, and says "wait here"
> when a room is full.**

## 🧠 Mental Model: Service Discovery

> **Service Discovery is like a phone book for microservices. Services register themselves
> ("I'm the orders-service, I'm at 10.0.1.42:8080"). Other services look up their
> address at runtime instead of hard-coding IPs that change on every deploy.**


---
## API Gateway — WHY, WHAT, HOW, WHEN

### WHY it exists

Without an API gateway in a microservices architecture, every client must:
- Know the address of every service
- Handle auth, rate limiting, and TLS on its own
- Deal with service-to-service protocol differences
- Hardcode addresses that change on every deploy

```
❌ WITHOUT API Gateway (direct client-to-service):
   Mobile app → auth-service:3001/login
   Mobile app → products-service:3002/products
   Mobile app → orders-service:3003/orders
   Mobile app → payments-service:3004/charge
   Mobile app → user-service:3005/profile
   → 5 different endpoints, 5 different auth configs, no central rate limiting
   → Mobile app must handle service discovery itself
   → Adding TLS to a new service = update all clients

✅ WITH API Gateway:
   Mobile app → api.shopflow.com/v1/login        → auth-service
   Mobile app → api.shopflow.com/v1/products     → products-service
   Mobile app → api.shopflow.com/v1/orders       → orders-service
   → ONE endpoint, one TLS cert, one auth config
   → Services can be rearranged without client changes
```

### WHAT an API Gateway does

```
Cross-cutting concerns (don't belong in business logic):
├── Authentication/Authorization   (verify JWT, check permissions)
├── Rate Limiting                  (100 req/min per user)
├── SSL/TLS Termination           (HTTPS → HTTP internally)
├── Request/Response Transformation (JSON ↔ Protobuf)
├── Load Balancing                 (distribute across service instances)
├── Circuit Breaking               (stop cascading failures)
├── Request Logging & Tracing      (add X-Request-ID, log latency)
├── Caching                        (cache GET responses at the edge)
├── Request Aggregation/BFF        (one gateway call → N service calls)
└── API Versioning                 (route /v1 vs /v2 to different backends)
```


In [ ]:
import time, uuid, re
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class Request:
    method: str
    path: str
    headers: dict = field(default_factory=dict)
    body: dict = field(default_factory=dict)

@dataclass
class Response:
    status: int
    body: dict
    headers: dict = field(default_factory=dict)

# ── Service registry (simplified) ────────────────────────────────────────────
ROUTES = {
    r"/v1/products.*": "products-service",
    r"/v1/orders.*":   "orders-service",
    r"/v1/users.*":    "users-service",
    r"/v1/auth.*":     "auth-service",
}

SERVICES = {
    "products-service": lambda req: Response(200, {"products": ["widget", "gadget"]}),
    "orders-service":   lambda req: Response(200, {"orders": [{"id": "ORD-001"}]}),
    "users-service":    lambda req: Response(200, {"user": {"id": "u1", "name": "Ada"}}),
    "auth-service":     lambda req: Response(200, {"token": "jwt.xxx.yyy"}),
}

class APIGateway:
    '''Simplified API Gateway: routing + auth + rate limiting + request ID injection.'''

    def __init__(self):
        self._rate_limits: dict[str, list[float]] = {}
        self._rate_limit_window = 60.0
        self._rate_limit_max    = 5   # 5 req/min for demo

    def _inject_request_id(self, req: Request) -> str:
        req_id = str(uuid.uuid4())[:8]
        req.headers["X-Request-ID"] = req_id
        return req_id

    def _check_auth(self, req: Request) -> Response | None:
        if req.path.startswith("/v1/auth"): return None  # auth endpoint is public
        token = req.headers.get("Authorization", "")
        if not token.startswith("Bearer "):
            return Response(401, {"error": "Missing auth token"})
        return None

    def _check_rate_limit(self, client_ip: str) -> Response | None:
        now = time.monotonic()
        hits = self._rate_limits.setdefault(client_ip, [])
        # Sliding window: keep only hits within the window
        self._rate_limits[client_ip] = [t for t in hits if now - t < self._rate_limit_window]
        if len(self._rate_limits[client_ip]) >= self._rate_limit_max:
            return Response(429, {"error": f"Rate limit: {self._rate_limit_max} req/min exceeded",
                                   "retry_after": int(self._rate_limit_window)})
        self._rate_limits[client_ip].append(now)
        return None

    def _route(self, req: Request) -> str | None:
        for pattern, service in ROUTES.items():
            if re.match(pattern, req.path):
                return service
        return None

    def handle(self, req: Request, client_ip: str = "127.0.0.1") -> Response:
        req_id = self._inject_request_id(req)
        print(f"[GW] {req_id} {req.method} {req.path} from {client_ip}")

        # 1. Rate limit
        if (err := self._check_rate_limit(client_ip)): return err
        # 2. Auth
        if (err := self._check_auth(req)):             return err
        # 3. Route
        service = self._route(req)
        if not service:
            return Response(404, {"error": f"No route for {req.path}"})
        # 4. Forward to service
        resp = SERVICES[service](req)
        resp.headers["X-Request-ID"] = req_id
        resp.headers["X-Service"]    = service
        print(f"[GW] {req_id} → {service} → {resp.status}")
        return resp

# ── Demo ─────────────────────────────────────────────────────────────────────
gw = APIGateway()
print("=== API Gateway Demo ===")

# Normal authenticated request
r = gw.handle(Request("GET", "/v1/products/123",
                       headers={"Authorization": "Bearer valid-jwt"}))
print(f"Response: {r.status} {r.body}")
print()

# Unauthenticated request
r = gw.handle(Request("GET", "/v1/orders"))
print(f"Unauth: {r.status} {r.body}")
print()

# Rate limit test (6 requests, limit is 5)
print("Rate limiting (5 req/min limit):")
for i in range(6):
    r = gw.handle(Request("GET", f"/v1/products/{i}",
                           headers={"Authorization": "Bearer jwt"}),
                  client_ip="1.2.3.4")
    print(f"  Request {i+1}: HTTP {r.status}")


---
## Service Discovery — WHY, WHAT, HOW, WHEN

### WHY it exists

In a static deployment, services have fixed IPs. In a dynamic cloud environment:
- Services scale up/down (new instances = new IPs)
- Containers restart with new IPs
- Blue-green deployments swap entire fleets

**Without service discovery: hardcoded IPs break every deploy.**

### Two Patterns: Client-Side vs Server-Side Discovery

```
CLIENT-SIDE DISCOVERY           SERVER-SIDE DISCOVERY
────────────────────────        ────────────────────────
Client queries registry         Client queries a load balancer
Client picks an instance        LB queries registry internally
Client calls instance directly  LB forwards to chosen instance

Orders-service → Registry       Orders-service → LB → Registry
  "where is payments-service?"    "send to payments-service"
  "10.0.1.42:8080"               [LB picks internally]
  calls 10.0.1.42:8080           LB calls 10.0.1.42:8080

✓ No extra hop                  ✓ Simpler client
✗ Client must know registry     ✓ Works with any protocol
Example: Consul client, Eureka   Example: AWS ALB, Kubernetes DNS
```

### Self-Registration vs Third-Party Registration

```
Self-registration: service POSTs to registry on startup, sends heartbeats
  → Simple; service controls its own record
  → Risk: service dies without deregistering (stale entries)
  → Fix: TTL-based entries expire automatically

Third-party (platform) registration: Kubernetes registers pods automatically
  → Service code doesn't know about registry
  → Platform is responsible for accuracy
  → Used by Kubernetes, ECS, Nomad
```

### 🌍 Where Used in Production

| System | Type | Notes |
|---|---|---|
| Netflix Eureka | Self-registration | Java services; each registers on startup |
| HashiCorp Consul | Self + platform | DNS + HTTP; health checks built-in |
| Kubernetes DNS | Platform | `payments-service.default.svc.cluster.local` auto-created |
| AWS Cloud Map | Platform | Integrates with ECS, EKS, Lambda |
| Istio/Envoy | Sidecar | Service mesh intercepts all traffic; discovery via xDS API |

### ⚠️ Gotchas

- **Stale entries**: service crashes without deregistering. Fix: TTL + heartbeat
- **Registry becomes a SPOF**: run 3+ registry instances (Raft consensus for consistency)
- **DNS caching**: clients cache DNS for 30s+ even after service changes address
- **Health checks lag**: a service marked healthy may actually be unhealthy for seconds


In [ ]:
import time, threading, random
from dataclasses import dataclass, field

@dataclass
class ServiceInstance:
    service_name: str
    instance_id:  str
    host:         str
    port:         int
    metadata:     dict = field(default_factory=dict)
    last_heartbeat: float = field(default_factory=time.monotonic)
    healthy:      bool = True

class ServiceRegistry:
    '''Simplified service registry with health check and TTL-based expiry.'''
    TTL = 30.0   # seconds; instance removed if no heartbeat in 30s

    def __init__(self):
        self._instances: dict[str, list[ServiceInstance]] = {}
        self._lock = threading.Lock()

    def register(self, inst: ServiceInstance) -> None:
        with self._lock:
            self._instances.setdefault(inst.service_name, [])
            # Remove old entry for same instance_id if re-registering
            self._instances[inst.service_name] = [
                i for i in self._instances[inst.service_name]
                if i.instance_id != inst.instance_id
            ]
            self._instances[inst.service_name].append(inst)
            print(f"  [Registry] Registered: {inst.service_name}/{inst.instance_id} at {inst.host}:{inst.port}")

    def deregister(self, service_name: str, instance_id: str) -> None:
        with self._lock:
            self._instances[service_name] = [
                i for i in self._instances.get(service_name, [])
                if i.instance_id != instance_id
            ]
            print(f"  [Registry] Deregistered: {service_name}/{instance_id}")

    def heartbeat(self, service_name: str, instance_id: str) -> None:
        with self._lock:
            for inst in self._instances.get(service_name, []):
                if inst.instance_id == instance_id:
                    inst.last_heartbeat = time.monotonic(); return

    def discover(self, service_name: str) -> list[ServiceInstance]:
        '''Returns healthy instances; evicts expired ones.'''
        now = time.monotonic()
        with self._lock:
            instances = self._instances.get(service_name, [])
            live = [i for i in instances if now - i.last_heartbeat < self.TTL and i.healthy]
            self._instances[service_name] = live
            return live

    def pick(self, service_name: str) -> ServiceInstance | None:
        '''Client-side load balancing: random selection from healthy instances.'''
        instances = self.discover(service_name)
        return random.choice(instances) if instances else None

# ── Demo: dynamic scaling of payments-service ────────────────────────────────
registry = ServiceRegistry()
print("=== Service Discovery Demo ===")

# Three instances register at startup
for i in range(3):
    registry.register(ServiceInstance(
        service_name="payments-service",
        instance_id=f"pay-{i}",
        host=f"10.0.1.{10 + i}",
        port=8080,
        metadata={"version": "2.1.0", "region": "us-east-1"}
    ))

# Orders-service discovers and calls payments-service
print("
Orders-service discovering payments-service:")
for _ in range(5):
    inst = registry.pick("payments-service")
    print(f"  → Routed to {inst.instance_id} at {inst.host}:{inst.port}")

# Simulate: pay-1 crashes without deregistering (stale entry after TTL expires)
print("
Simulating pay-1 crash (no deregister, TTL will expire)...")
for inst in registry._instances["payments-service"]:
    if inst.instance_id == "pay-1":
        inst.last_heartbeat = time.monotonic() - 35   # fake: heartbeat 35s ago (past TTL)

print("After TTL expiry, discover() returns only live instances:")
live = registry.discover("payments-service")
print(f"  Live instances: {[i.instance_id for i in live]} (pay-1 auto-removed)")

# Scale out: add pay-3
registry.register(ServiceInstance("payments-service", "pay-3", "10.0.1.13", 8080))
print(f"After scale-out: {[i.instance_id for i in registry.discover('payments-service')]}")
